# Session 3: simulate a winter, hour by hour
**A. Real-world problem.** Session 2 used daily average temperatures and ignored the house's
thermal mass, the capacity of the machine and its backup heater. Real winters have cold
nights, and a house cools slowly. Today you simulate a whole heating season hour by hour,
check the program against energy conservation and a hand calculation, test how insulation,
supply temperature and tariff change the bill, and find out what happens on the coldest night
when the heat pump is too small.

Educational model only, not an installation design.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
ETA = 0.45            # real COP as a fraction of Carnot (typical)
COP_MAX = 8.0
DEFROST_BELOW_C = 2.0  # below this outdoor temperature frost builds up on the outdoor coil
DEFROST_FACTOR = 0.90  # simple derate of the COP while that happens (illustrative)
TARIFF = 0.25          # illustrative money units per kWh

## Step 1: the machine
COP = eta x Carnot (kelvin), capped, and multiplied by a defrost factor below 2 C.
Heating capacity also falls in the cold: here 2 percent per kelvin below 7 C (illustrative),
never below 30 percent of the rated value (rated = capacity at 7 C outdoors).

In [ ]:
def cop_hp(t_out_c, t_sup_c, eta=ETA):
    t_out = np.asarray(t_out_c, dtype=float)
    lift = np.maximum(t_sup_c - t_out, 5.0)
    cop = np.minimum(eta * (t_sup_c + 273.15) / lift, COP_MAX)
    return np.where(t_out < DEFROST_BELOW_C, cop * DEFROST_FACTOR, cop)


def capacity_kw(t_out_c, rated_kw):
    return rated_kw * np.maximum(1.0 + 0.02 * (np.asarray(t_out_c, dtype=float) - 7.0), 0.3)

## Step 2: the weather
Seeded synthetic winter: seasonal cosine (-3 C in mid season, 9 C at the ends), a daily
swing of 3 K (coldest at 03:00), and day-to-day weather noise (an AR(1) series with a
fixed seed, so every run gives the same winter).

In [ ]:
def weather(n_days=151, seed=7):
    rng = np.random.default_rng(seed)
    noise = np.zeros(n_days + 1)
    for i in range(1, n_days + 1):
        noise[i] = 0.7 * noise[i - 1] + rng.normal(0.0, 2.4)
    h = np.arange(24 * n_days)
    seasonal = 3.0 + 6.0 * np.cos(2 * np.pi * h / (24 * n_days))
    daily = 3.0 * np.cos(2 * np.pi * ((h % 24) - 15) / 24)
    return seasonal + daily + np.interp(h / 24.0, np.arange(n_days + 1), noise)


def cold_week(hours=168):
    """A deterministic cold week (no random numbers) used by the live app: coldest near hour 100."""
    h = np.arange(hours, dtype=float)
    return -2.0 + 3.0 * np.cos(2 * np.pi * (h - 15) / 24) - 6.0 * np.exp(-(((h - 100) / 30) ** 2))

## Step 3: the house as one thermal capacity (an RC model)
C dT/dt = Q_heat - UA (T - T_out), with C in kWh per K and UA in kW per K. The time constant
is tau = C / UA hours. For heat Q held constant during one step the exact solution is

    T_next = T_eq + (T - T_eq) exp(-dt / tau),     T_eq = T_out + Q / UA

so the simulation is stable for any step. The thermostat asks for the heat that would bring
the house to the set point at the end of the step (never negative). The heat pump supplies up
to its capacity; a backup resistance heater (up to backup_kw) supplies the rest.

In [ ]:
def simulate(t_out, ua=0.20, cap_c=4.0, t_set=20.0, t_sup=35.0, rated_kw=9.0, backup_kw=9.0,
             eta=ETA, tariff=TARIFF, mode="heat_pump", dt=1.0, t0=None):
    """Hourly simulation. mode = 'heat_pump' or 'heater' (resistance heater only, max backup_kw)."""
    t_out = np.asarray(t_out, dtype=float)
    n = len(t_out)
    tau = cap_c / ua
    e = np.exp(-dt / tau)
    t_in = np.empty(n + 1)
    t_in[0] = t_set if t0 is None else t0
    q_hp, q_bu, elec, t_mean = np.zeros(n), np.zeros(n), np.zeros(n), np.zeros(n)
    cop = cop_hp(t_out, t_sup, eta)
    cap = capacity_kw(t_out, rated_kw)
    for i in range(n):
        t_eq_wanted = (t_set - t_in[i] * e) / (1.0 - e)
        demand = max(ua * (t_eq_wanted - t_out[i]), 0.0)
        if mode == "heat_pump":
            q_hp[i] = min(demand, cap[i])
            q_bu[i] = min(demand - q_hp[i], backup_kw)
        else:
            q_bu[i] = min(demand, backup_kw)
        q = q_hp[i] + q_bu[i]
        t_eq = t_out[i] + q / ua
        t_in[i + 1] = t_eq + (t_in[i] - t_eq) * e
        t_mean[i] = t_eq + (t_in[i] - t_eq) * (tau / dt) * (1.0 - e)   # mean indoor T during the hour
        elec[i] = (q_hp[i] / cop[i] + q_bu[i]) * dt
    heat = float(((q_hp + q_bu) * dt).sum())
    total = float(elec.sum())
    return dict(t_in=t_in, t_out=t_out, q_hp=q_hp, q_bu=q_bu, elec=elec, cop=cop, t_mean=t_mean,
                heat_kwh=heat, elec_kwh=total, cost=total * tariff, backup_kwh=float((q_bu * dt).sum()),
                min_t_in=float(t_in.min()), seasonal_cop=heat / total if total > 0 else float("nan"),
                ua=ua, cap_c=cap_c, dt=dt)

## Step 4: validate the simulation
1. **Energy conservation.** Stored energy change C (T_end - T_start) must equal the heat
   supplied minus the heat lost through the walls, sum of UA (T_mean - T_out) dt.
2. **Steady state by hand.** With constant outdoor temperature and a big enough machine the
   house settles at the set point; the heat needed is UA (T_set - T_out) and the electricity
   is that divided by the COP.

In [ ]:
def energy_residual(res):
    """Stored energy change minus (heat in - heat lost), in kWh. Should be about 0."""
    dt = res["dt"]
    stored = res["cap_c"] * (res["t_in"][-1] - res["t_in"][0])
    supplied = float(((res["q_hp"] + res["q_bu"]) * dt).sum())
    lost = float((res["ua"] * (res["t_mean"] - res["t_out"]) * dt).sum())
    return stored - (supplied - lost)


def steady_electric_kw(ua, t_set, t_out_c, t_sup_c, eta=ETA):
    """Hand calculation: electric power when the house sits at the set point."""
    return ua * (t_set - t_out_c) / float(cop_hp(t_out_c, t_sup_c, eta))

## Step 5: the season, the sensitivity and the failure case

In [ ]:
def size_rated_kw(ua, t_set, t_design_c):
    """Rated (7 C) capacity so that capacity at the design temperature equals the heat loss."""
    return ua * (t_set - t_design_c) / float(capacity_kw(t_design_c, 1.0))


if __name__ == "__main__":
    T = weather()
    print(f"winter: {len(T)} hours, mean {T.mean():.2f} C, coldest {T.min():.1f} C at hour {int(T.argmin())}")
    rated = size_rated_kw(0.20, 20.0, -10.0)
    hp = simulate(T, rated_kw=rated)
    ht = simulate(T, mode="heater", backup_kw=20.0)
    print(f"rated size for -10 C design: {rated:.2f} kW")
    print(f"heater   : {ht['elec_kwh']:.0f} kWh, cost {ht['cost']:.0f}, min indoor {ht['min_t_in']:.2f} C")
    print(f"heat pump: {hp['elec_kwh']:.0f} kWh, cost {hp['cost']:.0f}, seasonal COP {hp['seasonal_cop']:.2f}, "
          f"backup {hp['backup_kwh']:.1f} kWh, min indoor {hp['min_t_in']:.2f} C")
    print(f"energy residual: {energy_residual(hp):.2e} kWh")
    st = simulate(np.zeros(400), rated_kw=9.0)
    print(f"steady state at 0 C: sim {st['elec'][-1]:.4f} kW, hand {steady_electric_kw(0.2, 20.0, 0.0, 35.0):.4f} kW")
    print("sensitivity (season cost, heat pump / heater):")
    for ua in (0.12, 0.20, 0.30):
        a = simulate(T, ua=ua, rated_kw=size_rated_kw(ua, 20.0, -10.0))
        b = simulate(T, ua=ua, mode="heater", backup_kw=20.0)
        print(f"  UA {ua:.2f}: {a['cost']:.0f} / {b['cost']:.0f}")
    for ts in (30.0, 35.0, 45.0, 55.0):
        a = simulate(T, t_sup=ts, rated_kw=rated)
        print(f"  supply {ts:.0f} C: cost {a['cost']:.0f}, seasonal COP {a['seasonal_cop']:.2f}")
    for tar in (0.15, 0.25, 0.40):
        print(f"  tariff {tar:.2f}: heat pump {hp['elec_kwh'] * tar:.0f}, heater {ht['elec_kwh'] * tar:.0f}")
    # failure: undersized pump on the coldest night
    i0 = int(T.argmin())
    sl = slice(max(i0 - 48, 0), i0 + 24)
    for label, kw in (("sized", rated), ("undersized 5 kW, backup 9 kW", 5.0)):
        r = simulate(T[sl], rated_kw=kw if kw == rated else 5.0)
        print(f"  {label}: min indoor {r['min_t_in']:.2f} C, backup {r['backup_kwh']:.1f} kWh, elec {r['elec_kwh']:.1f} kWh")
    r = simulate(T[sl], rated_kw=5.0, backup_kw=0.0)
    print(f"  undersized 5 kW, no backup: min indoor {r['min_t_in']:.2f} C")

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    # Figure 1: the coldest days of the winter, sized and undersized
    sl = slice(max(i0 - 72, 0), i0 + 48)
    hrs = np.arange(sl.start, sl.stop)
    fig, axs = plt.subplots(2, 1, figsize=(8.5, 6), sharex=True)
    axs[0].plot(hrs, T[sl], color="#2a6fa8", label="outdoor")
    for label, kw, bu, col in ((f"indoor, sized {rated:.1f} kW", rated, 9.0, "#2a9d8f"), ("indoor, 5 kW and no backup", 5.0, 0.0, "#d9534f")):
        r = simulate(T[sl], rated_kw=kw, backup_kw=bu)
        axs[0].plot(hrs, r["t_in"][1:], color=col, label=label)
    axs[0].set_ylabel("temperature (C)")
    axs[0].set_title("The coldest days: a small heat pump without backup loses the set point")
    axs[0].legend(fontsize=8)
    axs[0].grid(alpha=0.3)
    r = simulate(T[sl], rated_kw=5.0)
    axs[1].stackplot(hrs, r["q_hp"], r["q_bu"], labels=["heat pump heat", "backup resistance heat"], colors=["#2a9d8f", "#d9534f"])
    axs[1].set_xlabel("hour of the winter")
    axs[1].set_ylabel("heat delivered (kW)")
    axs[1].set_title("5 kW heat pump with a 9 kW backup: the backup takes over on the coldest night")
    axs[1].legend(fontsize=8, loc="upper left")
    axs[1].grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "coldest_nights.png", dpi=120)

    # Figure 2: validation
    fig2, axs2 = plt.subplots(1, 2, figsize=(10, 4))
    outs = np.linspace(-10, 12, 12)
    sim_kw = [simulate(np.full(300, t), rated_kw=15.0)["elec"][-1] for t in outs]
    axs2[0].plot(outs, [steady_electric_kw(0.20, 20.0, t, 35.0) for t in outs], "-", label="hand calculation")
    axs2[0].plot(outs, sim_kw, "o", ms=5, label="simulation (after 300 h)")
    axs2[0].set_xlabel("constant outdoor temperature (C)")
    axs2[0].set_ylabel("electric power (kW)")
    axs2[0].set_title("Steady state: simulation versus UA (T_set - T_out) / COP")
    axs2[0].legend(fontsize=8)
    axs2[0].grid(alpha=0.3)
    uas = np.linspace(0.1, 0.4, 7)
    sim_e = [simulate(T, ua=u, rated_kw=size_rated_kw(u, 20.0, -10.0))["elec_kwh"] for u in uas]
    axs2[1].plot(uas, sim_e, "o-", label="heat pump electricity")
    axs2[1].set_xlabel("UA (kW per K)")
    axs2[1].set_ylabel("winter electricity (kWh)")
    axs2[1].set_title("Better insulation (smaller UA) cuts the bill")
    axs2[1].grid(alpha=0.3)
    axs2[1].legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "validation_and_ua.png", dpi=120)

## Where this stops
One lumped capacity, perfect set-point control, a fixed supply temperature, no hot water, no
solar or internal gains, an invented capacity curve and defrost factor, and a synthetic
winter. Real design needs a certified calculation and local weather data.

## Coding tasks
1. **Easy.** Run the winter with `t_sup=45.0` and print the seasonal COP; compare it with 3.72 at 35 C.
2. **Stretch.** Add a night setback (17 C from 22:00 to 06:00) by making `t_set` an array and compare the winter cost.
3. **Explain.** Explain, with the time constant, why the simulation can use a one-hour step even though the thermostat changes its mind every hour.

## Assessment
- **A. Conceptual.** On a single very cold night a house with a large heat capacity needs less backup heat than a lightweight house with the same UA. Explain why, and say why it does not help in a long cold spell.
- **B. Numerical.** If the heating fails completely while it is 0 C outdoors, UA = 0.20 kW/K and C = 4.0 kWh/K, how long does the house take to cool from 20 C to 15 C?
- **C. Graph and simulation interpretation.** In the figure, the 5 kW heat pump with backup shows the backup heater switching on in the coldest hours. Estimate the capacity of the 5 kW pump at -11 C and the backup power needed, if the house loss at that temperature is 6.2 kW.
- **D. Design and optimisation challenge.** Design: a poorly insulated house has UA = 0.30 kW/K, 20 C set point, 45 C supply, design temperature -10 C, and a 9 kW backup. Choose a rated capacity: compare the design-temperature size with 10 kW and 8 kW using the seeded winter.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    T = L.weather()
    checks.append(("weather is reproducible (same seed, same winter)", np.array_equal(T, L.weather())))
    checks.append(("weather has 151 x 24 = 3624 hours", len(T) == 3624))
    checks.append(("the coldest hour is between -13 and -9 C", -13.0 < T.min() < -9.0))

    # exact solution with no heating at all: T = T_out + (T0 - T_out) exp(-t / tau)
    r = L.simulate(np.zeros(48), ua=0.2, cap_c=4.0, rated_kw=0.0, backup_kw=0.0)
    exact = 20.0 * np.exp(-np.arange(49) / (4.0 / 0.2))
    checks.append(("a house with no heating cools like exp(-t / tau), tau = C / UA = 20 h", np.allclose(r["t_in"], exact, atol=1e-9)))

    # energy conservation, three different situations
    rated = L.size_rated_kw(0.20, 20.0, -10.0)
    hp = L.simulate(T, rated_kw=rated)
    ht = L.simulate(T, mode="heater", backup_kw=20.0)
    sag = L.simulate(T[1600:1900], rated_kw=5.0, backup_kw=0.0)
    checks.append(("energy conservation: heat pump season residual below 1e-6 kWh", abs(L.energy_residual(hp)) < 1e-6))
    checks.append(("energy conservation: heater season residual below 1e-6 kWh", abs(L.energy_residual(ht)) < 1e-6))
    checks.append(("energy conservation holds when the house cools (undersized, no backup)", abs(L.energy_residual(sag)) < 1e-6))

    # steady state against the hand calculation
    st = L.simulate(np.zeros(300), rated_kw=9.0)
    hand = 0.20 * 20.0 / (0.45 * 308.15 / 35.0 * 0.9)
    checks.append((f"worked example: steady state at 0 C: heat 0.20 x 20 = 4.00 kW, COP 3.962 x 0.9 = 3.566, electricity 1.122 kW (got {st['elec'][-1]:.4f})",
                   abs(st["elec"][-1] - hand) < 1e-4 and abs(st["elec"][-1] - L.steady_electric_kw(0.20, 20.0, 0.0, 35.0)) < 1e-9))
    st2 = L.simulate(np.full(300, 5.0), rated_kw=9.0, dt=0.5)
    checks.append(("steady state does not depend on the time step (dt = 0.5 h)", abs(st2["elec"][-1] / 0.5 - L.steady_electric_kw(0.20, 20.0, 5.0, 35.0)) < 1e-4))
    checks.append(("no heat is needed when it is as warm outside as the set point", L.simulate(np.full(24, 20.0))["elec_kwh"] < 1e-9))

    # machine model
    checks.append(("defrost derate: COP just below 2 C is 0.9 times the COP just above", abs(float(L.cop_hp(1.99, 35.0)) / float(L.cop_hp(2.0, 35.0)) - 0.9 * (35 - 2.0) / (35 - 1.99)) < 1e-9))
    checks.append(("capacity at the design temperature equals the sizing target (6 kW)", abs(float(L.capacity_kw(-10.0, rated)) - 6.0) < 1e-9))
    checks.append(("capacity never falls below 30 percent of rated", float(L.capacity_kw(-60.0, 10.0)) == 3.0))

    # season results
    checks.append(("a heater needs electricity equal to the heat it delivers", abs(ht["elec_kwh"] - ht["heat_kwh"]) < 1e-9))
    checks.append((f"heat pump seasonal COP is between 3 and 4.5 (got {hp['seasonal_cop']:.2f})", 3.0 < hp["seasonal_cop"] < 4.5))
    checks.append(("heat pump costs less than 30 percent of the heater", hp["cost"] < 0.30 * ht["cost"]))
    checks.append(("a properly sized heat pump holds the set point all winter", hp["min_t_in"] > 19.99))

    # sensitivity
    c = [L.simulate(T, ua=u, rated_kw=L.size_rated_kw(u, 20.0, -10.0))["cost"] for u in (0.12, 0.20, 0.30)]
    checks.append(("sensitivity: more heat loss (larger UA) costs more", c[0] < c[1] < c[2]))
    s = [L.simulate(T, t_sup=t, rated_kw=rated)["cost"] for t in (30.0, 35.0, 45.0, 55.0)]
    checks.append(("sensitivity: a higher supply temperature costs more", s[0] < s[1] < s[2] < s[3]))
    checks.append(("sensitivity: cost is proportional to the tariff", abs(hp["elec_kwh"] * 0.5 - L.simulate(T, rated_kw=rated, tariff=0.5)["cost"]) < 1e-9))

    # failure case: undersized pump on the coldest night
    i0 = int(T.argmin())
    sl = slice(i0 - 48, i0 + 24)
    small = L.simulate(T[sl], rated_kw=5.0, backup_kw=0.0)
    backed = L.simulate(T[sl], rated_kw=5.0, backup_kw=9.0)
    good = L.simulate(T[sl], rated_kw=rated, backup_kw=9.0)
    checks.append((f"failure: a 5 kW heat pump without backup lets the house fall below 15 C (got {small['min_t_in']:.1f})", small["min_t_in"] < 15.0))
    checks.append(("failure fixed by backup, at a cost: the backup holds 20 C but uses more electricity", backed["min_t_in"] > 19.99 and backed["elec_kwh"] > 1.4 * good["elec_kwh"]))

    # mini-project reference
    mp = L.simulate(T, ua=0.26, t_sup=45.0, rated_kw=8.5, backup_kw=6.0)
    mp_small = L.simulate(T, ua=0.26, t_sup=45.0, rated_kw=8.0, backup_kw=6.0)
    checks.append(("mini-project reference: 8.5 kW has backup share below 2 percent, 8.0 kW does not",
                   mp["backup_kwh"] / mp["heat_kwh"] < 0.02 < mp_small["backup_kwh"] / mp_small["heat_kwh"]))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
